# Torun Reservoir WAQ — FPV scenario 02 diagnosis with attribution (Baseline · 08 · FPV 02)

Self-contained diagnosis of the corrected FPV scenario 02 against the corrected baseline, using the
attribution run **08** to separate the two ways the panels act on the water-quality model.

| Run | FLOW hydrodynamics (`.vol .flo .tem .vdf …`) | WAQ forcing (`RadSurf`/`RadAve`, `VWind`) | Role |
|---|---|---|---|
| **Baseline** | no FPV | no FPV | reference |
| **08** | no FPV (baseline files) | FPV 02 (× 0.65 under the panels) | shading + wind inside WAQ only |
| **FPV 02** | FPV 02 | FPV 02 | full FPV effect as modelled |

Three differences are reported everywhere. They add up exactly: Δ total = Δ forcing + Δ hydro.

| Name | Difference | What it isolates |
|---|---|---|
| **Δ total** | FPV 02 − Baseline | everything the panels change in the model |
| **Δ forcing** | 08 − Baseline | less light and wind in WAQ, identical hydrodynamics |
| **Δ hydro** | FPV 02 − 08 | the FPV FLOW run (water level, temperature, mixing), identical WAQ forcing |

**Stations.** CAB (on the edge of the footprint), BIS (outside it) and **P3**, a column deep inside the
footprint chosen automatically in Section 3 (or fixed with `P3_MN`).

**Data sources.**
- CAB and BIS: the `.his` files (every substance and diagnostic, including `Chlfa`, `SecchiDept`, `Limit*`).
- P3 and the maps: the NetCDF files converted from `.map`. They hold the 25 state variables only, so there is
  **no `Chlfa`, Secchi or `Limit*` at P3 or in the maps** — total algal biomass (gC/m³, sum of the BLOOM
  ecotypes) is used there instead, and for CAB and BIS as well wherever the three stations are compared.
- Layer thickness, water depth and temperature: the communication files each run read (`.vol`/`SURF` from the
  segment-order `.par`, `.tem`), read once a day at 00:00 (the BLOOM time step).
- Footprint and covered fraction: the baseline and FPV radiation `.bin` files.

**Conventions.**
- Period: 1 April – 31 August 2023. Values are **daily means** of the 6-hourly output (as in the thesis tables),
  so they include the 00:00 BLOOM step.
- Layers: `.his` label `CAB (N)` is WAQ layer `K = N − 1`; the first wet layer of the stations is K = 7.
- Depth axes of the profiles: nominal mid-depth of each layer, from the baseline June–August mean layer
  thicknesses of that column (the real top-cell thickness changes with the water level — see Section 5).
- Δ colour scales are symmetric (95th/98th percentile of |Δ|); absolute scales pool the runs shown (2–98th percentile).

**Sections**
1. Configuration · 2. Readers and helpers · 3. Grid, footprint and stations · 4. Consistency checks ·
5. Water level and top cell · 6. Station time series · 7. Depth profiles · 8. Chlorophyll a and Secchi (CAB, BIS) ·
9. Phytoplankton groups · 10. BLOOM limitation flags (CAB, BIS) · 11. Spatial analysis ·
12. Chl a and total algae, three runs side by side (maps, footprint/lake/P3–BIS profiles) · 13. Summary tables ·
14. Reading the results

## 1. Configuration
Edit this cell only. Everything else is derived from it.

In [ ]:
import os, struct, time, warnings
from pathlib import Path
from functools import lru_cache

import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.gridspec as gridspec
from matplotlib.patches import Patch
from scipy import ndimage

warnings.filterwarnings('ignore', category=RuntimeWarning)

# ---- folders ------------------------------------------------------------------------------------------
MODEL_DIR   = Path(r"<MODEL_DIR>")          # e.g. //luna/.../delft_modelling/wq_0.1min_secondfolder
HIS_DIR     = MODEL_DIR / 'his_scenario_files_remmaped'
MAP_DIR     = MODEL_DIR / 'map_files_scenarios_remmaped'
RESULTS_DIR = MODEL_DIR / 'results_fpv_diagnosis'
LOOKUP      = MODEL_DIR / 'segment_lookup.npz'      # segnum (M, N, K) and new_to_old
CLEAN       = 'com-WQexport_clean'                  # prefix of the trimmed communication files
PAR_NAME    = 'Torun_WAQ_Apr2023_segmentorder.par'  # SURF, bottomdept per segment (segment order)

# ---- runs ---------------------------------------------------------------------------------------------
RAD_NOFPV  = MODEL_DIR / 'waq-sw_radiation_flux-noFPV_remapped_upper.bin'
WIND_NOFPV = MODEL_DIR / 'waq-wind-noFPV_remapped_upper.bin'
RAD_02     = MODEL_DIR / 'waq-daily-sw_radiation_flux-02towaq_fpv41_nogaps_remapped_upper.bin'
WIND_02    = MODEL_DIR / 'waq-wind-02towaq_fpv41_nogaps_remapped_upper.bin'

RUNS = {
    'Base': dict(label='Baseline',
                 his=HIS_DIR / 'Torun_WAQ_Apr2023.his',
                 nc=MAP_DIR / 'Torun_WAQ_Apr2023.nc',
                 hyd=MODEL_DIR / '01_baseline_rebuilt',
                 rad=RAD_NOFPV, wind=WIND_NOFPV),
    '08':   dict(label='08 (baseline FLOW + FPV forcing)',
                 his=MODEL_DIR / '08_baselinewithSRWSfrom02' / 'Torun_WAQ_Apr2023.his',
                 nc=MAP_DIR / 'Torun_WAQ_Apr2023_scen08.nc',
                 hyd=MODEL_DIR / '08_baselinewithSRWSfrom02',
                 rad=RAD_02, wind=WIND_02),
    'FPV':  dict(label='FPV 02',
                 his=HIS_DIR / 'Torun_WAQ_Apr2023_scen02.his',
                 nc=MAP_DIR / 'Torun_WAQ_Apr2023_scen02.nc',
                 hyd=MODEL_DIR / '02scen_fixed',
                 rad=RAD_02, wind=WIND_02),
}
# Differences: name -> (run, reference run, label)
DIFFS = {
    'total':   ('FPV', 'Base', 'Δ total (FPV − Baseline)'),
    'forcing': ('08',  'Base', 'Δ forcing (08 − Baseline)'),
    'hydro':   ('FPV', '08',   'Δ hydro (FPV − 08)'),
}
RUN_STYLE = {'Base': dict(color='steelblue', ls='-',  lw=1.2),
             '08':   dict(color='0.35',      ls='--', lw=1.1),
             'FPV':  dict(color='tomato',    ls='-',  lw=1.2)}

# ---- time ---------------------------------------------------------------------------------------------
T0        = pd.Timestamp('2022-09-09')   # reference time of all files
SIM_START = pd.Timestamp('2023-04-01')
SIM_END   = pd.Timestamp('2023-08-31')   # last day analysed (inclusive)
SUMMER    = (pd.Timestamp('2023-06-01'), pd.Timestamp('2023-08-31'))

# ---- stations (trimmed 1-based segment numbers from the .inp, top to bottom) ----------------------------
STATIONS = {
    'CAB': dict(segments=[1114, 2456, 3772, 5050, 6283, 7486, 8626, 9693, 10630], his='CAB'),
    'BIS': dict(segments=[1635, 2974, 4283, 5543, 6766, 7964, 9078, 10121, 10973], his='BIS'),
}
P3_MN = None          # (M, N) of the third station; None = deepest-inside cell of the footprint (Section 3)

# ---- model constants (from the .inp) ------------------------------------------------------------------
MIN_DEPTH   = 0.1     # MinDepth (m)
FULL_COVER  = 0.65    # radiation/wind factor under full panel cover
ALGAE = ['FDIATOMS_E', 'FDIATOMS_N', 'FDIATOMS_P', 'FFLAGELA', 'GREENS_E', 'GREENS_N', 'GREENS_P',
         'MICROCYS_E', 'MICROCYS_N', 'MICROCYS_P']
GROUPS = {'Diatoms': 'FDIATOMS', 'Flagellates': 'FFLAGELA', 'Greens': 'GREENS', 'Cyanobacteria': 'MICROCYS'}
GROUP_COLOURS = ['#ff7f0e', '#1f77b4', '#2ca02c', '#9467bd']

assert "<MODEL_DIR>" not in str(MODEL_DIR), 'Set MODEL_DIR first'
RESULTS_DIR.mkdir(exist_ok=True)
SIM_END_EXCL = SIM_END + pd.Timedelta(days=1)
DAYS = pd.date_range(SIM_START, SIM_END, freq='D')
print(f'Model folder: {MODEL_DIR}\nPeriod: {SIM_START.date()} – {SIM_END.date()} ({len(DAYS)} days)')
print(f'Results to: {RESULTS_DIR}')

## 2. Readers and helpers
File formats: `.his` = 160-character title, `nvar`, `nloc`, 20-character names, then records
`[int32 time, float32 (nloc × nvar)]`; segment files (`.vol`, `.tem`, forcing `.bin`) = records
`[int32 time, float32 × N_SEG]`; `.par` = header + (SURF, bottomdept) per segment. Times are seconds after `T0`.

In [ ]:
# ---- .his ----------------------------------------------------------------------------------------------
def load_his(path):
    """Read a DELWAQ .his file, keep the analysis window. -999 -> NaN."""
    with open(path, 'rb') as f:
        f.read(160)
        nvar, nloc = struct.unpack('<2i', f.read(8))
        names = [f.read(20).decode('latin-1').strip() for _ in range(nvar)]
        locs = []
        for _ in range(nloc):
            f.read(4)
            locs.append(f.read(20).decode('latin-1').strip())
        rec = np.dtype([('t', '<i4'), ('v', '<f4', (nloc, nvar))])
        n = (os.path.getsize(path) - f.tell()) // rec.itemsize
        data = np.fromfile(f, dtype=rec, count=n)
    t = T0 + pd.to_timedelta(data['t'].astype('int64'), unit='s')
    keep = (t >= SIM_START) & (t < SIM_END_EXCL)
    v = data['v'][keep].astype(float)
    v[v <= -998] = np.nan
    print(f'  {Path(path).name}: {nvar} variables, {nloc} locations, {n} records ({keep.sum()} in window)')
    return dict(path=Path(path), names=names, idx={s: i for i, s in enumerate(names)},
                locs=locs, loc_idx={l: i for i, l in enumerate(locs)}, t=pd.DatetimeIndex(t[keep]), v=v)

def his_raw(R, loc, var):
    """Raw (6-hourly) series of var at location loc, or None if either is missing."""
    if loc not in R['loc_idx'] or var not in R['idx']:
        return None
    return pd.Series(R['v'][:, R['loc_idx'][loc], R['idx'][var]], index=R['t'])

def daily(s):
    """Daily mean on the analysis days (NaN where missing)."""
    if s is None or len(s) == 0:
        return pd.Series(np.nan, index=DAYS)
    s = s.loc[(s.index >= SIM_START) & (s.index < SIM_END_EXCL)]
    return s.resample('D').mean().reindex(DAYS)

# ---- segment files -------------------------------------------------------------------------------------
def seg_memmap(path, nseg):
    rec = np.dtype([('t', '<i4'), ('v', '<f4', (nseg,))])
    n = os.path.getsize(path) // rec.itemsize
    return np.memmap(path, dtype=rec, mode='r', shape=(n,))

def rows_at(mm, times):
    """Record indices of mm at the given Timestamps (fixed time step assumed, then verified)."""
    t0 = int(mm[0]['t']); dt = int(mm[1]['t']) - t0
    target = np.array([(t - T0).total_seconds() for t in times], dtype='int64')
    idx = (target - t0) // dt
    ok = (idx >= 0) & (idx < len(mm)) & ((target - t0) % dt == 0)
    idx = np.where(ok, idx, 0)
    got = np.array([int(mm[i]['t']) for i in idx])
    ok &= got == target
    return idx, ok

def read_daily(path, nseg, days=DAYS):
    """(n_days, nseg) float32 array of the records at 00:00 of each day (NaN where absent)."""
    mm = seg_memmap(path, nseg)
    idx, ok = rows_at(mm, days)
    out = np.full((len(days), nseg), np.nan, dtype='f4')
    if ok.any():
        out[ok] = mm[idx[ok]]['v']
    if not ok.all():
        print(f'  WARNING {Path(path).name}: {(~ok).sum()} of {len(days)} days have no record at 00:00')
    return out

def read_surf(folder):
    a = np.fromfile(Path(folder) / PAR_NAME, '<f4')
    assert a.size == 1 + 2 * N_SEG, f'{PAR_NAME}: {a.size - 1} values, expected {2 * N_SEG}'
    return a[1:].reshape(N_SEG, 2)[:, 0].astype(float)

# ---- plotting helpers (same look as the earlier diagnostic notebook) -------------------------------------
xlim = (SIM_START, SIM_END)
xfmt = mdates.DateFormatter('%b\n%Y')
xloc = mdates.MonthLocator()

def fmt_time(ax, size=8):
    ax.set_xlim(xlim); ax.xaxis.set_major_formatter(xfmt)
    ax.xaxis.set_major_locator(xloc); ax.tick_params(labelsize=size)

def tag(ax, text, y=0.97, size=9):
    ax.text(0.02, y, text, transform=ax.transAxes, fontsize=size, ha='left', va='top',
            bbox=dict(facecolor='white', edgecolor='none', pad=2.0, alpha=0.85))

def sym_lim(*arrays, q=95):
    v = np.concatenate([np.abs(np.asarray(a, float)).ravel() for a in arrays])
    v = v[np.isfinite(v)]
    lim = np.percentile(v, q) if v.size else 1.0
    return lim if lim > 0 else 1.0

def abs_lim(*arrays):
    v = np.concatenate([np.asarray(a, float).ravel() for a in arrays])
    v = v[np.isfinite(v)]
    return tuple(np.percentile(v, [2, 98])) if v.size else (0.0, 1.0)

def save(fig, name):
    fig.savefig(RESULTS_DIR / name, dpi=150, bbox_inches='tight')

print('helpers loaded')

### Load the outputs
`.his` for all three runs, NetCDF lazily (values are read when needed), and the daily volumes,
temperatures and `SURF` of the communication files each run read.

In [ ]:
lk = np.load(LOOKUP)
segnum, new_to_old = lk['segnum'], lk['new_to_old']
N_SEG = len(new_to_old)
NM, NN, NK = segnum.shape
print(f'segnum {segnum.shape}, {N_SEG:,} trimmed segments')

HIS, NC, VOL, TEM, SURF = {}, {}, {}, {}, {}
for r, cfg in RUNS.items():
    print(f'\n{r}: {cfg["label"]}')
    HIS[r] = load_his(cfg['his'])
    NC[r] = xr.open_dataset(cfg['nc']).sel(time=slice(SIM_START, SIM_END_EXCL - pd.Timedelta('1s')))
    print(f'  {Path(cfg["nc"]).name}: {NC[r].sizes["time"]} times, {len(NC[r].data_vars)} variables')
    SURF[r] = read_surf(cfg['hyd'])
    VOL[r] = read_daily(Path(cfg['hyd']) / f'{CLEAN}.vol', N_SEG)
    tem = Path(cfg['hyd']) / f'{CLEAN}.tem'
    TEM[r] = read_daily(tem, N_SEG) if tem.exists() else None
    print(f'  hydrodynamics: {cfg["hyd"].name} (daily .vol{" and .tem" if TEM[r] is not None else ""})')
THICK = {r: VOL[r] / SURF[r][None, :] for r in RUNS}    # layer thickness (m), (day, segment)
ALGAE = [a for a in ALGAE if a in NC['Base'] or a in HIS['Base']['idx']]
print(f'\nalgal state variables used: {ALGAE}')

## 3. Grid, footprint and stations
- Each trimmed segment gets its grid position (M, N, K) from `segnum` and `new_to_old`.
- The **covered fraction** of each column comes from the radiation files: `factor = FPV / baseline` on summer
  days, `covered = (1 − factor) / (1 − 0.65)`. The footprint is every column with a covered fraction > 0.
- **P3** is the fully covered column farthest from the footprint edge (ties: the deepest column).

In [ ]:
valid = ~np.isnan(segnum)
mm_, nn_, kk_ = np.nonzero(valid)
pos_old = np.full((int(np.nanmax(segnum)) + 1, 3), -1)
pos_old[segnum[valid].astype(int) - 1] = np.c_[mm_, nn_, kk_]
POS = pos_old[new_to_old]                                   # (N_SEG, 3): M, N, K of each trimmed segment
assert (POS >= 0).all(), 'some trimmed segments have no grid position'
SEG_OF = np.full((NM, NN, NK), -1)
SEG_OF[POS[:, 0], POS[:, 1], POS[:, 2]] = np.arange(N_SEG)

def column(m, n):
    """Wet layers K and their trimmed (0-based) segments for column (m, n), top to bottom."""
    ks = [k for k in range(NK) if SEG_OF[m, n, k] >= 0]
    return ks, [int(SEG_OF[m, n, k]) for k in ks]

# ---- covered fraction from the radiation forcing ----
def covered_fraction():
    mb, mf = seg_memmap(RUNS['Base']['rad'], N_SEG), seg_memmap(RUNS['FPV']['rad'], N_SEG)
    days = pd.date_range(*SUMMER, freq='10D')
    ib, okb = rows_at(mb, days); i_f, okf = rows_at(mf, days)
    ok = okb & okf
    b, f = mb[ib[ok]]['v'].astype(float), mf[i_f[ok]]['v'].astype(float)
    with np.errstate(invalid='ignore', divide='ignore'):
        ratio = np.where(b > 1, f / b, np.nan)
    factor = np.nanmedian(ratio, axis=0)                     # per segment
    seg_cov = np.clip((1 - factor) / (1 - FULL_COVER), 0, 1)
    seg_cov[np.abs(1 - factor) < 1e-4] = 0.0
    cov = np.zeros((NM, NN))
    np.maximum.at(cov, (POS[:, 0], POS[:, 1]), np.nan_to_num(seg_cov))
    return cov, ok.sum()

COVER, n_used = covered_fraction()
FOOT = COVER > 0
WET_MN = (SEG_OF >= 0).any(axis=2)
print(f'covered fraction from {n_used} summer days: footprint {FOOT.sum()} of {WET_MN.sum()} wet columns; '
      f'fully covered {(COVER >= 0.999).sum()}; classes {sorted(set(np.round(COVER[FOOT], 2)))}')

# ---- stations ----
ST = {}
for name, cfg in STATIONS.items():
    segs = [s - 1 for s in cfg['segments']]
    m, n = POS[segs[0], :2]
    ks, col_segs = column(m, n)
    assert col_segs == segs, f'{name}: .inp segments differ from the column at (M={m}, N={n}): {col_segs}'
    ST[name] = dict(m=int(m), n=int(n), ks=ks, segs=segs, his=cfg['his'])

full = COVER >= 0.999
dist = ndimage.distance_transform_edt(full)
if P3_MN is None:
    cand = np.argwhere(dist == dist.max())
    m3, n3 = max(cand, key=lambda c: (len(column(*c)[0]), -c[0], -c[1]))
else:
    m3, n3 = P3_MN
ks3, segs3 = column(m3, n3)
ST['P3'] = dict(m=int(m3), n=int(n3), ks=ks3, segs=segs3, his=None)

dist_out = ndimage.distance_transform_edt(~FOOT)            # distance to the footprint from outside
print(f'\n{"station":8s} {"(M, N)":>10s} {"layers K":>10s} {"covered":>8s} {"cells to edge":>14s}  his')
for name, s in ST.items():
    d_edge = dist[s['m'], s['n']] if FOOT[s['m'], s['n']] else -dist_out[s['m'], s['n']]
    s['cover'] = COVER[s['m'], s['n']]
    layers = f'{s["ks"][0]}-{s["ks"][-1]}'
    print(f'{name:8s} {str((s["m"], s["n"])):>10s} {layers:>10s} '
          f'{s["cover"]:8.2f} {d_edge:14.1f}  {s["his"] or "-"}')
print('cells to edge: > 0 inside the footprint (fully covered part), < 0 outside')

# ---- map of the footprint, covered fraction and stations ----
STN_MARK = {'CAB': 's', 'BIS': 'o', 'P3': '^'}
def mark_stations(ax, size=8):
    for name, s in ST.items():
        ax.plot(s['m'], s['n'], STN_MARK[name], ms=size, mfc='none', mec='k', mew=1.5)
        ax.annotate(name, (s['m'], s['n']), xytext=(5, 5), textcoords='offset points', fontsize=8)

def draw_footprint(ax):
    ax.contour(FOOT.T.astype(float), levels=[0.5], colors='k', linewidths=1.2)

fig, ax = plt.subplots(figsize=(6, 5))
pc = ax.pcolormesh(np.ma.masked_where(~WET_MN, COVER).T, cmap='YlOrBr', vmin=0, vmax=1, shading='nearest')
draw_footprint(ax); mark_stations(ax)
fig.colorbar(pc, ax=ax, shrink=0.8, label='covered fraction (from radiation forcing)')
ax.set_aspect('equal'); ax.set_xlabel('M'); ax.set_ylabel('N'); ax.tick_params(labelsize=8)
tag(ax, 'FPV 02 footprint and stations')
save(fig, 'Map_footprint_stations.png'); plt.show()

### Station data access
One function serves all stations:
- CAB and BIS read the `.his` (`'CAB (N)'`, N = K + 1);
- P3 reads the NetCDF at its (M, N, K);
- `Depth` (layer thickness) and, at P3, `Temp` come from the communication files;
- `Algae` is the sum of the algal state variables; `Algae_col` is its column integral (gC/m²).

In [ ]:
NC_LAYER_DIM = 'K'

@lru_cache(maxsize=None)
def series(st, run, var, k):
    """Daily series of var at station st, layer K = k (0-based), run run."""
    s = ST[st]
    if k not in s['ks']:
        return pd.Series(np.nan, index=DAYS)
    seg = s['segs'][s['ks'].index(k)]
    if var == 'Depth':
        return pd.Series(THICK[run][:, seg].astype(float), index=DAYS)
    if var == 'Temp' and s['his'] is None:
        return pd.Series(np.nan if TEM[run] is None else TEM[run][:, seg].astype(float), index=DAYS)
    if var == 'Algae':
        parts = [series(st, run, a, k) for a in ALGAE]
        parts = [p for p in parts if p.notna().any()]
        return pd.concat(parts, axis=1).sum(axis=1, min_count=1) if parts else pd.Series(np.nan, index=DAYS)
    if s['his'] is not None:
        return daily(his_raw(HIS[run], f'{s["his"]} ({k + 1})', var))
    if var in NC[run]:
        da = NC[run][var].isel(M=s['m'], N=s['n']).sel({NC_LAYER_DIM: k}).load()
        return daily(pd.Series(da.values.astype(float), index=pd.DatetimeIndex(da['time'].values)))
    return pd.Series(np.nan, index=DAYS)

def top(st, run, var):
    return series(st, run, var, ST[st]['ks'][0])

def bottom(st, run, var):
    return series(st, run, var, ST[st]['ks'][-1])

def depth_avg(st, run, var):
    """Thickness-weighted mean over the wet layers of the column."""
    num = den = 0
    for k in ST[st]['ks']:
        c, h = series(st, run, var, k), series(st, run, 'Depth', k)
        ok = c.notna() & (h > 0)
        num = num + (c * h).where(ok, 0); den = den + h.where(ok, 0)
    return (num / den.replace(0, np.nan))

def column_int(st, run, var):
    """Column integral sum(conc × thickness), e.g. gC/m² for Algae."""
    tot = 0
    for k in ST[st]['ks']:
        tot = tot + (series(st, run, var, k) * series(st, run, 'Depth', k)).fillna(0)
    return tot

def water_depth(st, run):
    return sum(series(st, run, 'Depth', k).fillna(0) for k in ST[st]['ks'])

# nominal layer mid-depths per station (baseline June–August mean thickness)
for name, s in ST.items():
    h = np.array([series(name, 'Base', 'Depth', k).loc[SUMMER[0]:SUMMER[1]].mean() for k in s['ks']])
    s['mid'] = np.cumsum(h) - h / 2
    print(f'{name}: nominal layer mid-depths (m) ' + ', '.join(f'{d:.2f}' for d in s['mid']))

def matrix(st, run, var):
    """(DAYS, layers) matrix of var at station st."""
    return np.column_stack([series(st, run, var, k).values for k in ST[st]['ks']])

## 4. Consistency checks
Every check prints PASS/FAIL; a FAIL means the comparisons below do not mean what they are meant to.

In [ ]:
RESULTS = {}
def record(name, ok, detail=''):
    RESULTS[name] = (bool(ok), detail)
    print(f"[{'PASS' if ok else 'FAIL'}] {name}" + (f' - {detail}' if detail else ''))

print('4.1 Files (modification dates)')
for r, cfg in RUNS.items():
    for key in ('his', 'nc'):
        p = Path(cfg[key])
        print(f'   {r:5s} {key:4s} {time.ctime(os.path.getmtime(p))}  {p.name}')

print('\n4.2 NetCDF matches .his at CAB, top layer (OXY, raw 6-hourly values)')
s = ST['CAB']
for r in RUNS:
    h = his_raw(HIS[r], f'CAB ({s["ks"][0] + 1})', 'OXY')
    da = NC[r]['OXY'].isel(M=s['m'], N=s['n']).sel({NC_LAYER_DIM: s['ks'][0]}).load()
    n = pd.Series(da.values.astype(float), index=pd.DatetimeIndex(da['time'].values))
    common = h.index.intersection(n.index)
    dmax = float(np.nanmax(np.abs(h.loc[common].values - n.loc[common].values))) if len(common) else np.nan
    record(f'{r}: NetCDF == .his at CAB', len(common) > 10 and dmax < 1e-3,
           f'{len(common)} common times, max |diff| {dmax:.2e} g/m3')

print('\n4.3 Run 08 uses the baseline hydrodynamics')
record('08: daily volumes identical to the baseline', np.array_equal(VOL['08'], VOL['Base'], equal_nan=True))
if TEM['08'] is not None and TEM['Base'] is not None:
    record('08: daily temperatures identical to the baseline', np.array_equal(TEM['08'], TEM['Base'], equal_nan=True))
record('FPV: volumes differ from the baseline (own FLOW run)', not np.array_equal(VOL['FPV'], VOL['Base'], equal_nan=True))
for st in ('CAB', 'BIS'):
    for v in ('Temp', 'Depth'):
        a, b = top(st, '08', v), top(st, 'Base', v)
        record(f'08 == Baseline: {v} at {st} top layer (.his)', np.allclose(a, b, equal_nan=True),
               f'max |diff| {np.nanmax(np.abs(a - b)):.2e}')
a, b = top('CAB', 'Base', 'Depth'), daily(his_raw(HIS['Base'], f'CAB ({ST["CAB"]["ks"][0] + 1})', 'Depth'))
record('.his Depth == .vol/SURF at CAB top layer (daily)', np.nanmax(np.abs(a - b)) < 0.02,
       f'max |diff| {np.nanmax(np.abs(a - b)):.3f} m (daily mean of 6-hourly .his vs 00:00 .vol)')

print('\n4.4 Forcing seen by WAQ at the stations (ratio to baseline; expected 1 − 0.35 × covered fraction)')
def bin_station_ratio(var, st, run):
    mm_r, mm_b = seg_memmap(RUNS[run][var], N_SEG), seg_memmap(RUNS['Base'][var], N_SEG)
    days = pd.date_range(*SUMMER, freq='7D')
    ir, okr = rows_at(mm_r, days); ib, okb = rows_at(mm_b, days)
    ok = okr & okb
    seg = ST[st]['segs'][0]
    r_, b_ = mm_r[ir[ok]]['v'][:, seg].astype(float), mm_b[ib[ok]]['v'][:, seg].astype(float)
    keep = b_ > 0.01
    return float(np.median(r_[keep] / b_[keep])) if keep.any() else np.nan

for st, s in ST.items():
    expect = 1 - (1 - FULL_COVER) * s['cover']
    for r in ('08', 'FPV'):
        for var in ('rad', 'wind'):
            got = bin_station_ratio(var, st, r)
            record(f'{r}: {var} factor at {st}', abs(got - expect) < 0.01, f'{got:.3f} (expected {expect:.3f})')
    if s['his'] is not None:
        for r in ('08', 'FPV'):
            ratio = (top(st, r, 'RadAve') / top(st, 'Base', 'RadAve')).median()
            record(f'{r}: RadAve ratio in .his at {st}', abs(ratio - expect) < 0.01, f'{ratio:.3f}')

n_fail = sum(not ok for ok, _ in RESULTS.values())
print(f'\n{len(RESULTS) - n_fail} PASS, {n_fail} FAIL')

## 5. Water level and top cell
The FPV FLOW run evaporates less under the panels, so its water level is higher. In this z-layer grid the
**top cell** is whatever lies between the water surface and the next fixed layer boundary, so a few
centimetres of level change its thickness a lot. WAQ skips processes in segments thinner than `MinDepth`.
- (a) total water volume, (b) the volume difference, (c) top-cell thickness at the three stations.

In [ ]:
vol_tot = {r: pd.Series(np.nansum(VOL[r], axis=1), index=DAYS) for r in RUNS}

fig = plt.figure(figsize=(11, 3.6))
gs = gridspec.GridSpec(1, 3, figure=fig, wspace=0.28)
ax = fig.add_subplot(gs[0, 0])
for r in ('Base', 'FPV'):
    ax.plot(DAYS, vol_tot[r] / 1e6, label=RUNS[r]['label'], **RUN_STYLE[r])
ax.set_ylabel('Total volume (Mm³)', fontsize=9); ax.legend(fontsize=7, loc='best'); ax.grid(True, alpha=0.3)
fmt_time(ax); tag(ax, '(a)')
ax = fig.add_subplot(gs[0, 1])
dv = vol_tot['FPV'] - vol_tot['Base']
ax.plot(DAYS, dv / 1e3, color='k', lw=1.2)
ax.set_ylabel('Δ volume FPV − Baseline (10³ m³)', fontsize=9); ax.grid(True, alpha=0.3)
fmt_time(ax); tag(ax, '(b)')
ax = fig.add_subplot(gs[0, 2])
for st, mk in STN_MARK.items():
    for r in ('Base', 'FPV'):
        ax.plot(DAYS, top(st, r, 'Depth'), color=RUN_STYLE[r]['color'], lw=1.0,
                ls={'CAB': '-', 'BIS': ':', 'P3': '-.'}[st], label=f'{st} {RUNS[r]["label"]}')
ax.axhline(MIN_DEPTH, color='k', lw=0.8, ls='--')
ax.text(DAYS[-2], MIN_DEPTH, 'MinDepth ', fontsize=7, va='bottom', ha='right')
ax.set_ylabel('Top-cell thickness (m)', fontsize=9); ax.legend(fontsize=6, loc='upper left', ncol=2)
ax.grid(True, alpha=0.3); fmt_time(ax); tag(ax, '(c)', y=0.12)
save(fig, 'WaterLevel_TopCell.png'); plt.show()

print('Total volume and water depth (1st and 15th of each month)')
dates = pd.date_range(SIM_START, SIM_END, freq='SMS')
tbl = pd.DataFrame({'V base (Mm3)': vol_tot['Base'].reindex(dates) / 1e6,
                    'dV FPV-base (%)': (dv / vol_tot['Base'] * 100).reindex(dates)})
for st in ST:
    tbl[f'{st} dDepth (m)'] = (water_depth(st, 'FPV') - water_depth(st, 'Base')).reindex(dates)
print(tbl.round(3).to_string())

print(f'\nShare of days with the top cell thinner than MinDepth ({MIN_DEPTH} m), by month')
shr = {f'{st} {r}': (top(st, r, 'Depth') < MIN_DEPTH).groupby(DAYS.to_period('M')).mean()
       for st in ST for r in ('Base', 'FPV')}
shr = pd.DataFrame(shr); shr.index = shr.index.strftime('%b')
print(shr.round(2).to_string())

## 6. Station time series
Same layout for the three stations. Lines: Baseline (blue), 08 (grey dashed), FPV 02 (red).
- (a) top-cell thickness, (b) surface temperature, (c) surface DO, (d) bottom DO,
  (e) surface algal biomass, (f) column algal biomass, (g) surface PO4, (h) surface NO3,
  (i) surface Chl a and (j) Secchi depth (`.his` only, so empty at P3).

In [ ]:
PANELS_TS = [
    ('Top-cell thickness (m)',      lambda st, r: top(st, r, 'Depth')),
    ('Surface temperature (°C)',    lambda st, r: top(st, r, 'Temp')),
    ('DO surface (g/m³)',           lambda st, r: top(st, r, 'OXY')),
    ('DO bottom (g/m³)',            lambda st, r: bottom(st, r, 'OXY')),
    ('Algae surface (gC/m³)',       lambda st, r: top(st, r, 'Algae')),
    ('Algae column (gC/m²)',        lambda st, r: column_int(st, r, 'Algae')),
    ('PO4 surface (gP/m³)',         lambda st, r: top(st, r, 'PO4')),
    ('NO3 surface (gN/m³)',         lambda st, r: top(st, r, 'NO3')),
    ('Chl a surface (µg/L)',        lambda st, r: top(st, r, 'Chlfa')),
    ('Secchi depth (m)',            lambda st, r: daily(his_raw(HIS[r], f'{ST[st]["his"]}_averaged', 'SecchiDept'))
                                                  if ST[st]['his'] else pd.Series(np.nan, index=DAYS)),
]

def station_timeseries(st):
    fig, axes = plt.subplots(5, 2, figsize=(9, 11), sharex=True)
    for ax, (label, fn), letter in zip(axes.flat, PANELS_TS, 'abcdefghij'):
        drawn = False
        for r in RUNS:
            y = fn(st, r)
            if y.notna().any():
                ax.plot(DAYS, y.values, label=RUNS[r]['label'], **RUN_STYLE[r]); drawn = True
        if label.startswith('Top-cell'):
            ax.axhline(MIN_DEPTH, color='k', lw=0.7, ls=':')
        if label.startswith('Secchi'):
            ax.invert_yaxis()
        ax.set_ylabel(label, fontsize=8); ax.grid(True, alpha=0.3); fmt_time(ax)
        tag(ax, f'({letter})' + ('' if drawn else '  not available at this station'), y=0.95)
    axes[0, 0].legend(fontsize=7, loc='upper right')
    for ax in axes[-1]:
        plt.setp(ax.get_xticklabels(), visible=True)
    fig.suptitle(f'{st} — (M={ST[st]["m"]}, N={ST[st]["n"]}), covered fraction {ST[st]["cover"]:.2f}', fontsize=10)
    plt.tight_layout()
    save(fig, f'Timeseries_{st}.png'); plt.show()

for st in ST:
    station_timeseries(st)

## 7. Depth profiles (Hovmöller)
Two figures per station.
- **Main:** (a) Baseline, (b) FPV 02, (c) Δ total — absolute colour bar shared by (a) and (b), Δ colour bar for (c).
- **Attribution:** Δ forcing, Δ hydro and Δ total on **one** symmetric colour scale per row, so their sizes
  can be compared directly (a pale Δ forcing panel means a small effect, not missing data).

In [ ]:
PROFILE_VARS = [
    ('OXY',   'DO (g/m³)',             'viridis'),
    ('Algae', 'Algae (gC/m³)',         'YlGn'),
    ('NO3',   'NO3 (gN/m³)',           'YlGnBu'),
    ('NH4',   'NH4 (gN/m³)',           'YlOrRd'),
    ('PO4',   'PO4 (gP/m³)',           'viridis'),
    ('Si',    'Si (gSi/m³)',           'YlOrBr'),
]
LETTERS = 'abcdefghijklmnopqrstuvwxyz'

def profile_main(st):
    s = ST[st]; dep = s['mid']; rows = len(PROFILE_VARS)
    fig = plt.figure(figsize=(10, 2.4 * rows))
    gs = gridspec.GridSpec(rows, 6, figure=fig, width_ratios=[1, 1, 1, 0.035, 0.30, 0.035], wspace=0.10,
                           hspace=0.10, top=0.97, bottom=0.04)
    p = 0
    for row, (var, lab, cmap) in enumerate(PROFILE_VARS):
        B, F = matrix(st, 'Base', var), matrix(st, 'FPV', var)
        D = F - B
        vmin, vmax = abs_lim(B, F); dl = sym_lim(D)
        axB = fig.add_subplot(gs[row, 0]); axF = fig.add_subplot(gs[row, 1], sharey=axB)
        axD = fig.add_subplot(gs[row, 2], sharey=axB)
        pc = axB.pcolormesh(DAYS, dep, B.T, cmap=cmap, vmin=vmin, vmax=vmax, shading='nearest')
        axF.pcolormesh(DAYS, dep, F.T, cmap=cmap, vmin=vmin, vmax=vmax, shading='nearest')
        pd_ = axD.pcolormesh(DAYS, dep, D.T, cmap='RdBu_r', vmin=-dl, vmax=dl, shading='nearest')
        axB.invert_yaxis()
        axB.set_ylabel('Depth (m)', fontsize=9)
        for ax, txt in ((axB, 'Baseline'), (axF, 'FPV 02'), (axD, 'Δ total')):
            fmt_time(ax, 7); tag(ax, f'({LETTERS[p]}) {txt} — {var}', y=0.95, size=8); p += 1
            if row < rows - 1:
                plt.setp(ax.get_xticklabels(), visible=False)
        plt.setp(axF.get_yticklabels(), visible=False); plt.setp(axD.get_yticklabels(), visible=False)
        fig.colorbar(pc, cax=fig.add_subplot(gs[row, 3]), label=lab).ax.tick_params(labelsize=7)
        fig.colorbar(pd_, cax=fig.add_subplot(gs[row, 5]), label=f'Δ {lab}').ax.tick_params(labelsize=7)
    fig.suptitle(f'{st} — depth profiles, Baseline vs FPV 02', fontsize=10, y=0.995)
    save(fig, f'Profiles_main_{st}.png'); plt.show()

def profile_attribution(st):
    s = ST[st]; dep = s['mid']; rows = len(PROFILE_VARS)
    fig = plt.figure(figsize=(10, 2.4 * rows))
    gs = gridspec.GridSpec(rows, 4, figure=fig, width_ratios=[1, 1, 1, 0.035], wspace=0.10, hspace=0.10,
                           top=0.97, bottom=0.04)
    p = 0
    for row, (var, lab, _) in enumerate(PROFILE_VARS):
        M = {r: matrix(st, r, var) for r in RUNS}
        Ds = {d: M[a] - M[b] for d, (a, b, _) in DIFFS.items()}
        dl = sym_lim(*Ds.values())
        axes = []
        for c, d in enumerate(('forcing', 'hydro', 'total')):
            ax = fig.add_subplot(gs[row, c], sharey=axes[0] if axes else None)
            pc = ax.pcolormesh(DAYS, dep, Ds[d].T, cmap='RdBu_r', vmin=-dl, vmax=dl, shading='nearest')
            fmt_time(ax, 7); tag(ax, f'({LETTERS[p]}) {DIFFS[d][2]} — {var}', y=0.95, size=8); p += 1
            if row < rows - 1:
                plt.setp(ax.get_xticklabels(), visible=False)
            if c:
                plt.setp(ax.get_yticklabels(), visible=False)
            axes.append(ax)
        axes[0].invert_yaxis(); axes[0].set_ylabel('Depth (m)', fontsize=9)
        fig.colorbar(pc, cax=fig.add_subplot(gs[row, 3]), label=f'Δ {lab}').ax.tick_params(labelsize=7)
    fig.suptitle(f'{st} — attribution of the FPV change (one colour scale per row)', fontsize=10, y=0.995)
    save(fig, f'Profiles_attribution_{st}.png'); plt.show()

for st in ST:
    profile_main(st)
    profile_attribution(st)

## 8. Chlorophyll a and Secchi depth (CAB, BIS — `.his` only)
`Chlfa` is BLOOM's own chlorophyll, which the NetCDF does not contain. Secchi depth uses the `_averaged`
monitors (as in the thesis).

In [ ]:
def chla_profile(st):
    dep = ST[st]['mid']
    M = {r: matrix(st, r, 'Chlfa') for r in RUNS}
    D = {d: M[a] - M[b] for d, (a, b, _) in DIFFS.items()}
    vmin, vmax = abs_lim(M['Base'], M['FPV']); dl = sym_lim(*D.values(), q=98)
    fig = plt.figure(figsize=(12, 6))
    gs = gridspec.GridSpec(2, 5, figure=fig, left=0.06, right=0.96, top=0.95, bottom=0.10, hspace=0.30,
                           wspace=0.30, width_ratios=[1, 1, 0.05, 1, 0.05])
    ax_a = fig.add_subplot(gs[0, 0]); ax_b = fig.add_subplot(gs[0, 1], sharey=ax_a)
    ax_c = fig.add_subplot(gs[0, 3], sharey=ax_a)
    ax_d = fig.add_subplot(gs[1, 0], sharey=ax_a); ax_e = fig.add_subplot(gs[1, 1], sharey=ax_a)
    pc = ax_a.pcolormesh(DAYS, dep, M['Base'].T, cmap='YlGn', vmin=vmin, vmax=vmax, shading='nearest')
    ax_b.pcolormesh(DAYS, dep, M['FPV'].T, cmap='YlGn', vmin=vmin, vmax=vmax, shading='nearest')
    for ax, d in ((ax_c, 'total'), (ax_d, 'forcing'), (ax_e, 'hydro')):
        pdf = ax.pcolormesh(DAYS, dep, D[d].T, cmap='RdBu_r', vmin=-dl, vmax=dl, shading='nearest')
    ax_a.invert_yaxis()
    for ax, txt in ((ax_a, '(a) Baseline'), (ax_b, '(b) FPV 02'), (ax_c, f'(c) {DIFFS["total"][2]}'),
                    (ax_d, f'(d) {DIFFS["forcing"][2]}'), (ax_e, f'(e) {DIFFS["hydro"][2]}')):
        fmt_time(ax); tag(ax, txt, y=0.12)
    for ax in (ax_b, ax_c, ax_e):
        plt.setp(ax.get_yticklabels(), visible=False)
    ax_a.set_ylabel('Depth (m)', fontsize=9); ax_d.set_ylabel('Depth (m)', fontsize=9)
    fig.colorbar(pc, cax=fig.add_subplot(gs[0, 2]), label='Chlorophyll a (µg/L)')
    fig.colorbar(pdf, cax=fig.add_subplot(gs[0, 4]), label='Δ Chlorophyll a (µg/L)\n(same scale in c, d, e)')
    save(fig, f'Chla_profiles_{st}.png'); plt.show()
    print(f'{st} Chl a (µg/L), Apr–Aug mean:  surface / column mean')
    for r in RUNS:
        print(f'   {RUNS[r]["label"]:34s} {np.nanmean(M[r][:, 0]):7.3f} / {np.nanmean(M[r]):7.3f}')

def secchi_figure(st):
    S = {r: daily(his_raw(HIS[r], f'{ST[st]["his"]}_averaged', 'SecchiDept')) for r in RUNS}
    S = {r: s.where(s > 0) for r, s in S.items()}
    Mo = {r: s.resample('MS').mean() for r, s in S.items()}
    months = Mo['Base'].index; labels = [m.strftime('%b\n%y') for m in months]
    fig = plt.figure(figsize=(11, 4))
    gs = gridspec.GridSpec(1, 3, figure=fig, wspace=0.15)
    ax_a = fig.add_subplot(gs[0, 0]); ax_b = fig.add_subplot(gs[0, 1], sharey=ax_a); ax_c = fig.add_subplot(gs[0, 2])
    plt.setp(ax_b.get_yticklabels(), visible=False)
    for r in RUNS:
        ax_a.plot(DAYS, S[r].values, label=RUNS[r]['label'], **RUN_STYLE[r])
    ax_a.invert_yaxis(); ax_a.set_ylabel('Secchi depth (m)', fontsize=9); ax_a.legend(fontsize=7, loc='best')
    ax_a.set_title(f'(a) Daily Secchi depth ({st}_averaged)', fontsize=9, loc='left'); fmt_time(ax_a)
    ax_a.grid(True, alpha=0.3)
    x = np.arange(len(months)); w = 0.27
    for i, (r, col) in enumerate(zip(RUNS, ('steelblue', '0.6', 'tomato'))):
        ax_b.bar(x + (i - 1) * w, Mo[r].values, w, color=col, edgecolor='black', lw=0.4, label=RUNS[r]['label'])
    ax_b.set_xticks(x, labels=labels, fontsize=8); ax_b.legend(fontsize=7, loc='best')
    ax_b.set_title('(b) Monthly mean Secchi depth', fontsize=9, loc='left'); ax_b.grid(True, axis='y', alpha=0.3)
    dt_ = S['FPV'] - S['Base']
    ax_c.fill_between(DAYS, dt_.values, 0, where=dt_.values >= 0, color='blue', alpha=0.4, label='FPV clearer')
    ax_c.fill_between(DAYS, dt_.values, 0, where=dt_.values < 0, color='red', alpha=0.4, label='FPV murkier')
    ax_c.plot(DAYS, (S['08'] - S['Base']).values, color='k', lw=1.0, ls='--', label=DIFFS['forcing'][2])
    ax_c.axhline(0, color='k', lw=0.7, ls=':')
    ax_c.set_ylabel('Δ Secchi depth (m)', fontsize=9); ax_c.legend(fontsize=7, loc='best')
    ax_c.set_title('(c) Δ total (fill) and Δ forcing (line)', fontsize=9, loc='left'); fmt_time(ax_c)
    ax_c.grid(True, alpha=0.3)
    fig.subplots_adjust(left=0.07, right=0.98, top=0.85, bottom=0.18)
    pa, pb = ax_a.get_position(), ax_b.get_position()
    ax_b.set_position([pa.x0 + pa.width + 0.01, pb.y0, pb.width, pb.height])
    save(fig, f'Secchi_{st}_averaged.png'); plt.show()
    t = pd.DataFrame({RUNS[r]['label']: Mo[r] for r in RUNS})
    t['Δ total'] = t.iloc[:, 2] - t.iloc[:, 0]; t['Δ forcing'] = t.iloc[:, 1] - t.iloc[:, 0]
    t.index = t.index.strftime('%Y-%m')
    print(f'Secchi depth, monthly mean (m), {st}_averaged'); print(t.round(3).to_string())

for st in ('CAB', 'BIS'):
    chla_profile(st)
    secchi_figure(st)

## 9. Phytoplankton groups (surface layer)
(a) Baseline and (b) FPV 02 stacked group biomass; (c) monthly means (solid = Baseline, hatched = FPV 02);
(d) monthly composition; (e) Δ total and (f) Δ forcing per group.

In [ ]:
def group_df(st, run):
    cols = {}
    for g, prefix in GROUPS.items():
        parts = [top(st, run, a) for a in ALGAE if a.startswith(prefix)]
        cols[g] = pd.concat(parts, axis=1).sum(axis=1, min_count=1) if parts else pd.Series(np.nan, index=DAYS)
    return pd.DataFrame(cols).fillna(0)

def phyto_figure(st):
    G = {r: group_df(st, r) for r in RUNS}
    GM = {r: g.resample('MS').mean() for r, g in G.items()}
    months = GM['Base'].index; labels = [m.strftime('%b\n%y') for m in months]
    fig, axes = plt.subplots(3, 2, figsize=(9, 9))
    for ax, r, t in ((axes[0, 0], 'Base', '(a) Baseline'), (axes[0, 1], 'FPV', '(b) FPV 02')):
        ax.stackplot(DAYS, [G[r][g].values for g in GROUPS], labels=list(GROUPS), colors=GROUP_COLOURS, alpha=0.85)
        ax.set_ylabel('Biomass (gC/m³)', fontsize=9); ax.set_title(f'{t} — surface', fontsize=9, loc='left')
        fmt_time(ax); ax.grid(True, alpha=0.3)
    axes[0, 0].legend(fontsize=7, loc='upper left')
    ymax = max(axes[0, 0].get_ylim()[1], axes[0, 1].get_ylim()[1])
    axes[0, 0].set_ylim(0, ymax); axes[0, 1].set_ylim(0, ymax)
    x = np.arange(len(months)); w = 0.38
    ax = axes[1, 0]; bb = np.zeros(len(months)); bf = np.zeros(len(months))
    for g, col in zip(GROUPS, GROUP_COLOURS):
        ax.bar(x - w / 2, GM['Base'][g].values, w, bottom=bb, color=col, edgecolor='black', lw=0.6, alpha=0.95)
        ax.bar(x + w / 2, GM['FPV'][g].values, w, bottom=bf, color=col, edgecolor='black', lw=0.6, alpha=0.45,
               hatch='//')
        bb += GM['Base'][g].values; bf += GM['FPV'][g].values
    ax.legend(handles=[Patch(facecolor=c, edgecolor='black', lw=0.4, label=g) for g, c in zip(GROUPS, GROUP_COLOURS)]
              + [Patch(facecolor='grey', edgecolor='black', alpha=0.95, label='Baseline (solid)'),
                 Patch(facecolor='grey', edgecolor='black', alpha=0.45, hatch='//', label='FPV 02 (hatched)')],
              fontsize=7, loc='upper left')
    ax.set_xticks(x); ax.set_xticklabels(labels); ax.tick_params(labelsize=8)
    ax.set_ylabel('Mean biomass (gC/m³)', fontsize=9); ax.set_title('(c) Monthly mean biomass', fontsize=9, loc='left')
    ax.grid(True, axis='y', alpha=0.3)
    ax = axes[1, 1]; bb = np.zeros(len(months)); bf = np.zeros(len(months))
    tb = GM['Base'].sum(axis=1).replace(0, np.nan).values; tf = GM['FPV'].sum(axis=1).replace(0, np.nan).values
    for g, col in zip(GROUPS, GROUP_COLOURS):
        vb, vf = GM['Base'][g].values / tb * 100, GM['FPV'][g].values / tf * 100
        ax.bar(x - w / 2, vb, w, bottom=bb, color=col, edgecolor='black', lw=0.6, alpha=0.95)
        ax.bar(x + w / 2, vf, w, bottom=bf, color=col, edgecolor='black', lw=0.6, alpha=0.45, hatch='//')
        bb += np.nan_to_num(vb); bf += np.nan_to_num(vf)
    ax.set_xticks(x); ax.set_xticklabels(labels); ax.set_ylim(0, 105); ax.tick_params(labelsize=8)
    ax.set_ylabel('% of total biomass', fontsize=9); ax.set_title('(d) Monthly composition', fontsize=9, loc='left')
    ax.grid(True, axis='y', alpha=0.3)
    for ax, d, t in ((axes[2, 0], 'total', '(e)'), (axes[2, 1], 'forcing', '(f)')):
        a, b, lab = DIFFS[d]
        for g, col in zip(GROUPS, GROUP_COLOURS):
            ax.plot(DAYS, (G[a][g] - G[b][g]).values, color=col, lw=1.1, label=g)
        ax.axhline(0, color='k', lw=0.7, ls='--', alpha=0.5)
        ax.set_ylabel('Δ biomass (gC/m³)', fontsize=9); ax.set_title(f'{t} {lab}', fontsize=9, loc='left')
        fmt_time(ax); ax.grid(True, alpha=0.3)
    lim = max(abs(np.array(axes[2, 0].get_ylim())).max(), abs(np.array(axes[2, 1].get_ylim())).max())
    axes[2, 0].set_ylim(-lim, lim); axes[2, 1].set_ylim(-lim, lim)
    axes[2, 0].legend(fontsize=7, loc='upper left')
    fig.suptitle(f'{st} — phytoplankton groups, surface layer', fontsize=10)
    plt.tight_layout(); save(fig, f'Phyto_groups_{st}.png'); plt.show()

for st in ST:
    phyto_figure(st)

## 10. BLOOM limitation flags (CAB, BIS — `.his` only)
`Limit nit`, `Limit pho`, `Limit sil`, `Limit e`, `Limit gro`, `Limit mor` are **on/off flags**: 1 means that
constraint is the one binding BLOOM's solution at that time step, 0 means it is not. They are not efficiency
factors (a low value does not mean strong limitation). Their mean over a period is the **fraction of time the
constraint binds**. Outputs that are not 0/1 (e.g. `Limit Chlo`) are left out.
- (a)–(c): fraction of time each flag is on in the top layer, by month, for each run.
- Table: June–August fraction by layer.

In [ ]:
def limit_flags(R):
    out = []
    for v in R['names']:
        if v.startswith('Limit'):
            layer_locs = [i for l, i in R['loc_idx'].items() if 'averaged' not in l.lower()]
            vals = R['v'][:, layer_locs, R['idx'][v]]
            vals = vals[np.isfinite(vals)]
            if vals.size and np.isin(np.round(vals, 6), [0, 1]).all():
                out.append(v)
    return out

FLAGS = [f for f in limit_flags(HIS['Base']) if all(f in limit_flags(HIS[r]) for r in RUNS)]
print(f'on/off limitation flags: {FLAGS}')
FLAG_COLOURS = dict(zip(FLAGS, plt.cm.tab10(np.arange(len(FLAGS)))))

def limits_figure(st):
    k0 = ST[st]['ks'][0]
    fig, axes = plt.subplots(1, 3, figsize=(11, 3.5), sharey=True)
    for ax, r, letter in zip(axes, RUNS, 'abc'):
        mo = pd.DataFrame({f: series(st, r, f, k0) for f in FLAGS}).resample('MS').mean()
        x = np.arange(len(mo)); w = 0.8 / max(len(FLAGS), 1)
        for i, f in enumerate(FLAGS):
            ax.bar(x + (i - (len(FLAGS) - 1) / 2) * w, mo[f].values, w, color=FLAG_COLOURS[f],
                   edgecolor='black', lw=0.3, label=f)
        ax.set_xticks(x, labels=[m.strftime('%b\n%y') for m in mo.index], fontsize=8)
        ax.set_title(f'({letter}) {RUNS[r]["label"]}', fontsize=9, loc='left'); ax.grid(True, axis='y', alpha=0.3)
        ax.set_ylim(0, 1.05)
    axes[0].set_ylabel(f'Fraction of time binding\n{st} ({k0 + 1}) top layer', fontsize=9)
    axes[0].legend(fontsize=7, loc='upper left')
    plt.tight_layout(); save(fig, f'Limitation_flags_{st}.png'); plt.show()
    rows = {}
    for r in RUNS:
        for f in FLAGS:
            rows[(RUNS[r]['label'], f)] = {f'{st} ({k + 1})': series(st, r, f, k).loc[SUMMER[0]:SUMMER[1]].mean()
                                          for k in ST[st]['ks']}
    print(f'{st}: fraction of time binding, June–August, by layer')
    print(pd.DataFrame(rows).T.round(2).to_string())

for st in ('CAB', 'BIS'):
    limits_figure(st)

## 11. Spatial analysis (NetCDF, June–August mean)
- **11.1** Baseline | FPV 02 | Δ total maps for surface and near-surface algae, surface and bottom DO, and
  surface PO4 and NO3. "Bottom" is the deepest wet layer of each column.
- **11.2** Attribution maps in %: Δ forcing, Δ hydro and Δ total, one colour scale per row. Cells with less than
  1 % of the field's maximum are blank (percentages there are meaningless).
- **11.3** Mean % change inside vs outside the footprint, and at the three stations.
- **11.4** Water depth change and how often the top cell is thinner than `MinDepth`, from the `.vol` files.

In [ ]:
def nc_mean(run, var, k=None, bottom_layer=False):
    """June–August mean (M, N) field of var (or 'Algae') at layer k, or the deepest wet layer."""
    ds = NC[run].sel(time=slice(SUMMER[0], SUMMER[1] + pd.Timedelta(hours=23)))
    names = ALGAE if var == 'Algae' else [var]
    tot = None
    for v in names:
        if v not in ds:
            continue
        da = ds[v] if (k is None) else ds[v].sel({NC_LAYER_DIM: k})
        m = da.mean('time', skipna=True)
        tot = m if tot is None else tot + m
    if tot is None:
        return np.full((NM, NN), np.nan)
    if not bottom_layer:
        return tot.transpose('M', 'N').values.astype(float)
    a = tot.transpose('M', 'N', NC_LAYER_DIM).values.astype(float)
    last = np.where(np.isfinite(a).any(axis=2), a.shape[2] - 1 - np.argmax(np.isfinite(a[:, :, ::-1]), axis=2), -1)
    out = np.full((NM, NN), np.nan)
    ok = last >= 0
    out[ok] = a[np.nonzero(ok)[0], np.nonzero(ok)[1], last[ok]]
    return out

K_SURF = ST['CAB']['ks'][0]
MAP_VARS = [   # (key, var, layer, bottom?, label, cmap)
    ('Algae surface', 'Algae', K_SURF,     False, 'Algae (gC/m³)', 'YlGn'),
    ('Algae layer 2',  'Algae', K_SURF + 1, False, 'Algae (gC/m³)', 'YlGn'),
    ('DO surface',    'OXY',   K_SURF,     False, 'DO (g/m³)',     'viridis'),
    ('DO bottom',     'OXY',   None,       True,  'DO (g/m³)',     'viridis'),
    ('PO4 surface',   'PO4',   K_SURF,     False, 'PO4 (gP/m³)',   'viridis'),
    ('NO3 surface',   'NO3',   K_SURF,     False, 'NO3 (gN/m³)',   'YlGnBu'),
]
FIELDS = {key: {r: nc_mean(r, var, k, bot) for r in RUNS} for key, var, k, bot, _, _ in MAP_VARS}
print('fields computed:', list(FIELDS))

def map_axes(ax):
    ax.set_aspect('equal'); ax.tick_params(labelsize=7)
    draw_footprint(ax); mark_stations(ax, size=6)

def maps_main():
    rows = len(MAP_VARS)
    fig = plt.figure(figsize=(11, 3.0 * rows))
    gs = gridspec.GridSpec(rows, 6, figure=fig, width_ratios=[1, 1, 1, 0.035, 0.30, 0.035], wspace=0.12,
                           hspace=0.12, top=0.97, bottom=0.03)
    p = 0
    for row, (key, _, _, _, lab, cmap) in enumerate(MAP_VARS):
        B, F = FIELDS[key]['Base'], FIELDS[key]['FPV']
        D = F - B; vmin, vmax = abs_lim(B, F); dl = sym_lim(D, q=98)
        axs = [fig.add_subplot(gs[row, c]) for c in range(3)]
        pc = axs[0].pcolormesh(B.T, cmap=cmap, vmin=vmin, vmax=vmax, shading='nearest')
        axs[1].pcolormesh(F.T, cmap=cmap, vmin=vmin, vmax=vmax, shading='nearest')
        pdf = axs[2].pcolormesh(D.T, cmap='RdBu_r', vmin=-dl, vmax=dl, shading='nearest')
        for ax, t in zip(axs, ('Baseline', 'FPV 02', 'Δ total')):
            map_axes(ax); tag(ax, f'({LETTERS[p]}) {t} — {key}', y=0.98, size=8); p += 1
        fig.colorbar(pc, cax=fig.add_subplot(gs[row, 3]), label=lab).ax.tick_params(labelsize=7)
        fig.colorbar(pdf, cax=fig.add_subplot(gs[row, 5]), label=f'Δ {lab}').ax.tick_params(labelsize=7)
    fig.suptitle('June–August mean, Baseline vs FPV 02 (black outline = footprint)', fontsize=10, y=0.995)
    save(fig, 'Maps_main.png'); plt.show()

def pct(new, ref):
    out = np.full(ref.shape, np.nan)
    ok = np.isfinite(ref) & np.isfinite(new) & (np.abs(ref) > 0.01 * np.nanmax(np.abs(ref)))
    out[ok] = (new[ok] - ref[ok]) / ref[ok] * 100
    return out

def maps_attribution():
    rows = len(MAP_VARS)
    fig = plt.figure(figsize=(11, 3.0 * rows))
    gs = gridspec.GridSpec(rows, 4, figure=fig, width_ratios=[1, 1, 1, 0.035], wspace=0.12, hspace=0.12,
                           top=0.97, bottom=0.03)
    p = 0
    for row, (key, *_rest) in enumerate(MAP_VARS):
        P = {d: pct(FIELDS[key][a], FIELDS[key][b]) for d, (a, b, _) in DIFFS.items()}
        dl = sym_lim(*P.values(), q=98)
        for c, d in enumerate(('forcing', 'hydro', 'total')):
            ax = fig.add_subplot(gs[row, c])
            pc = ax.pcolormesh(P[d].T, cmap='RdBu_r', vmin=-dl, vmax=dl, shading='nearest')
            map_axes(ax); tag(ax, f'({LETTERS[p]}) {DIFFS[d][2]} — {key}', y=0.98, size=8); p += 1
        fig.colorbar(pc, cax=fig.add_subplot(gs[row, 3]), label='% change').ax.tick_params(labelsize=7)
    fig.suptitle('June–August mean % change — attribution (one colour scale per row)', fontsize=10, y=0.995)
    save(fig, 'Maps_attribution.png'); plt.show()

maps_main()
maps_attribution()

In [ ]:
rows = []
for key, *_ in MAP_VARS:
    for d, (a, b, lab) in DIFFS.items():
        P = pct(FIELDS[key][a], FIELDS[key][b])
        row = {'field': key, 'difference': lab,
               'inside footprint (%)': np.nanmean(P[FOOT]), 'outside (%)': np.nanmean(P[~FOOT & WET_MN]),
               'all (%)': np.nanmean(P[WET_MN])}
        for st, s in ST.items():
            row[f'{st} (%)'] = P[s['m'], s['n']]
        rows.append(row)
spatial_tbl = pd.DataFrame(rows).set_index(['field', 'difference'])
print('11.3 June–August mean % change (NetCDF fields)')
print(spatial_tbl.round(1).to_string())
spatial_tbl.to_csv(RESULTS_DIR / 'Spatial_attribution_summary.csv')

In [ ]:
# ---- 11.4 water depth and thin top cell from the .vol files ----
col_id = POS[:, 0] * NN + POS[:, 1]
summer = (DAYS >= SUMMER[0]) & (DAYS <= SUMMER[1])
top_seg = {}
for s_ in range(N_SEG):
    c = col_id[s_]
    if c not in top_seg or POS[s_, 2] < POS[top_seg[c], 2]:
        top_seg[c] = s_
top_cols = np.array(list(top_seg)); top_segs = np.array([top_seg[c] for c in top_cols])

def depth_map(run):
    wd = np.zeros((summer.sum(), NM * NN))
    for i, d in enumerate(np.flatnonzero(summer)):
        wd[i] = np.bincount(col_id, weights=np.nan_to_num(THICK[run][d]), minlength=NM * NN)
    return wd.mean(axis=0).reshape(NM, NN)

def thin_map(run):
    out = np.full(NM * NN, np.nan)
    out[top_cols] = (THICK[run][summer][:, top_segs] < MIN_DEPTH).mean(axis=0)
    return out.reshape(NM, NN)

WD = {r: depth_map(r) for r in ('Base', 'FPV')}
THIN = {r: thin_map(r) for r in ('Base', 'FPV')}
dWD = np.where(WET_MN, WD['FPV'] - WD['Base'], np.nan)

fig, axes = plt.subplots(1, 3, figsize=(13, 3.8))
ax = axes[0]
hi = np.nanmax(np.abs(dWD)) if np.isfinite(dWD).any() else 1.0
pc = ax.pcolormesh(dWD.T, cmap='Blues', vmin=0 if np.nanmin(dWD) >= 0 else -hi, vmax=hi * 1.05, shading='nearest')
map_axes(ax); tag(ax, '(a) Δ water depth FPV − Baseline', y=0.98, size=8)
fig.colorbar(pc, ax=ax, shrink=0.8, label='Δ water depth (m), Jun–Aug mean')
for ax, (r, letter) in zip(axes[1:], (('Base', 'b'), ('FPV', 'c'))):
    pc2 = ax.pcolormesh(np.ma.masked_invalid(THIN[r]).T, cmap='Reds', vmin=0, vmax=1, shading='nearest')
    map_axes(ax); tag(ax, f'({letter}) {RUNS[r]["label"]}: top cell < MinDepth', y=0.98, size=8)
    fig.colorbar(pc2, ax=ax, shrink=0.8, label='fraction of days, Jun–Aug')
plt.tight_layout()
save(fig, 'Maps_waterdepth_topcell.png'); plt.show()
print(f'Δ water depth (m), Jun–Aug mean over wet columns: mean {np.nanmean(dWD):.3f}, '
      f'range {np.nanmin(dWD):.3f} to {np.nanmax(dWD):.3f}')
print(f'columns with the top cell < MinDepth on > 50 % of summer days: '
      f'Baseline {(THIN["Base"] > 0.5).sum()}, FPV 02 {(THIN["FPV"] > 0.5).sum()} (of {WET_MN.sum()})')

## 12. Chlorophyll a and total algae — the three runs side by side
Each figure has the same six columns:
**Baseline | 08 | FPV 02** (one shared absolute colour bar) and **Δ forcing | Δ hydro | Δ total**
(one shared symmetric colour bar), so raw values and differences can be read against each other.

- **12.1** Daily fields of total algae and Chl a from the NetCDF (layers from the surface down), cached in
  `RESULTS_DIR` so later runs of this section are fast.
- **12.2** Surface-layer maps, mean over `CHL_PERIOD`, plus Δ forcing on its own colour scale (it is
  pale on the shared Δ scale because Δ hydro is much larger).
- **12.3** Depth profiles (Hovmöller), volume-weighted over (a) the FPV footprint and (b) the whole lake.
- **12.4** Depth profiles at P3 (under the panels) and BIS (outside), on common colour scales.

**Chl a outside the `.his` stations.** The NetCDF has no `Chlfa`. BLOOM computes `Chlfa` from the biomass of
each algal type times a fixed chlorophyll-to-carbon ratio per type, so the ratios can be recovered from the
`.his` (where both biomass and `Chlfa` exist) by non-negative least squares. 12.1 prints the fitted ratios and how
exactly they reproduce `Chlfa` in all three runs. If the error is not small, treat the derived Chl a as approximate.
(If some algal types always occur in fixed proportions, the individual ratios are not unique; only the
reproduction error matters.)

In [ ]:
from scipy.optimize import nnls

CHL_PERIOD = SUMMER                     # period averaged in the maps
KS = list(range(K_SURF, NK))            # layers analysed (surface layer first)

# ---- fit Chl a = sum(ratio_i × biomass_i) on the baseline .his, check on all runs ----
def his_matrix(R):
    locs = [i for l, i in R['loc_idx'].items() if 'averaged' not in l.lower()]
    X = R['v'][:, locs][:, :, [R['idx'][a] for a in ALGAE]].reshape(-1, len(ALGAE))
    y = R['v'][:, locs, R['idx']['Chlfa']].reshape(-1)
    ok = np.isfinite(X).all(axis=1) & np.isfinite(y)
    return X[ok], y[ok]

X, y = his_matrix(HIS['Base'])
CHL_COEF, _ = nnls(X, y)
print('Fitted Chl a / biomass ratios (µg Chl per gC, i.e. µg/L per gC/m³):')
for a, c in zip(ALGAE, CHL_COEF):
    print(f'   {a:12s} {c:8.2f}')
for r in RUNS:
    X, y = his_matrix(HIS[r])
    err = np.abs(X @ CHL_COEF - y)
    big = y > 1
    print(f'{RUNS[r]["label"]:34s}: max |error| {err.max():.3f} µg/L, '
          f'max relative error where Chlfa > 1 µg/L {np.max(err[big] / y[big]) * 100 if big.any() else np.nan:.2f} %')

# ---- daily fields (day, layer, M, N) ----
def daily_fields(run):
    cache = RESULTS_DIR / f'daily_algae_chl_{run}.npz'
    nc_time = os.path.getmtime(RUNS[run]['nc'])
    if cache.exists() and os.path.getmtime(cache) > nc_time:
        z = np.load(cache)
        if z['coef'].shape == CHL_COEF.shape and np.allclose(z['coef'], CHL_COEF):
            return z['alg'], z['chl']
    alg = chl = None
    for a, w in zip(ALGAE, CHL_COEF):
        da = NC[run][a].sel({NC_LAYER_DIM: KS}).transpose('time', NC_LAYER_DIM, 'M', 'N')
        d = da.resample(time='1D').mean().reindex(time=DAYS).values.astype('f4')
        alg = d if alg is None else alg + d
        chl = d * w if chl is None else chl + d * w
    np.savez_compressed(cache, alg=alg, chl=chl, coef=CHL_COEF)
    return alg, chl

FIELD3 = {}
for r in RUNS:
    t0 = time.time()
    FIELD3[r] = dict(zip(('Algae', 'Chl a'), daily_fields(r)))
    print(f'{r}: daily fields {FIELD3[r]["Algae"].shape} ({time.time() - t0:.0f} s)')

# layer volumes on the same grid (day, layer, M, N)
in_ks = POS[:, 2] >= K_SURF
def vol_grid(run):
    g = np.full((len(DAYS), len(KS), NM, NN), np.nan, 'f4')
    g[:, POS[in_ks, 2] - K_SURF, POS[in_ks, 0], POS[in_ks, 1]] = VOL[run][:, in_ks]
    return g
VGRID = {r: vol_grid(r) for r in RUNS}

VARS12 = [('Chl a', 'Chl a (µg/L)', 'YlGn'), ('Algae', 'Total algae (gC/m³)', 'YlGn')]
COLS12 = [('Base', 'Baseline'), ('08', '08 (FPV forcing)'), ('FPV', 'FPV 02')]
DCOLS12 = [('forcing', 'Δ forcing (08 − Base)'), ('hydro', 'Δ hydro (FPV − 08)'), ('total', 'Δ total (FPV − Base)')]

def six_columns(fig, rows, draw, ylabel=None):
    """rows: list of (row title, {run: array}, label, cmap). draw(ax, array, **kw) returns the mappable."""
    gs = gridspec.GridSpec(len(rows), 9, figure=fig, width_ratios=[1, 1, 1, 0.05, 0.22, 1, 1, 1, 0.05],
                           wspace=0.08, hspace=0.18, left=0.05, right=0.96, top=0.92, bottom=0.07)
    p = 0
    for i, (rt, data, lab, cmap) in enumerate(rows):
        D = {d: data[a] - data[b] for d, (a, b, _) in DIFFS.items()}
        vmin, vmax = abs_lim(*data.values()); dl = sym_lim(*D.values(), q=98)
        axes = []
        for c, (r, title) in enumerate(COLS12):
            ax = fig.add_subplot(gs[i, c], sharey=axes[0] if axes else None)
            pc = draw(ax, data[r], cmap=cmap, vmin=vmin, vmax=vmax); axes.append(ax)
            tag(ax, f'({LETTERS[p]}) {title}', y=0.98, size=8); p += 1
        for c, (d, title) in enumerate(DCOLS12):
            ax = fig.add_subplot(gs[i, 5 + c], sharey=axes[0])
            pd_ = draw(ax, D[d], cmap='RdBu_r', vmin=-dl, vmax=dl); axes.append(ax)
            tag(ax, f'({LETTERS[p]}) {title}', y=0.98, size=8); p += 1
        for ax in axes[1:]:
            plt.setp(ax.get_yticklabels(), visible=False)
        axes[0].set_ylabel(f'{rt}\n{ylabel}' if ylabel else rt, fontsize=9)
        if i < len(rows) - 1:
            for ax in axes:
                plt.setp(ax.get_xticklabels(), visible=False)
        fig.colorbar(pc, cax=fig.add_subplot(gs[i, 3]), label=lab).ax.tick_params(labelsize=7)
        fig.colorbar(pd_, cax=fig.add_subplot(gs[i, 8]), label=f'Δ {lab}').ax.tick_params(labelsize=7)

### 12.2 Surface layer maps
Mean over `CHL_PERIOD` of the surface layer (K = 7). Black outline: FPV footprint.

In [ ]:
def period_mean(a):
    sel = (DAYS >= CHL_PERIOD[0]) & (DAYS <= CHL_PERIOD[1])
    return np.nanmean(a[sel], axis=0)

def draw_map(ax, a, **kw):
    pc = ax.pcolormesh(np.ma.masked_invalid(a).T, shading='nearest', **kw)
    map_axes(ax)
    return pc

rows = [(f'{v}, surface', {r: np.where(WET_MN, period_mean(FIELD3[r][v][:, 0]), np.nan) for r in RUNS}, lab, cmap)
        for v, lab, cmap in VARS12]
fig = plt.figure(figsize=(19, 6.8))
six_columns(fig, rows, draw_map)
fig.suptitle(f'Surface layer, mean {CHL_PERIOD[0]:%d %b} – {CHL_PERIOD[1]:%d %b %Y}', fontsize=11)
save(fig, 'Compare3_maps_surface_Chla_Algae.png'); plt.show()

# Δ forcing on its own colour scale (it is small next to Δ hydro, so it is pale in the figure above)
fig, axes = plt.subplots(2, 2, figsize=(11, 8.5))
for i, (rt, data, lab, _) in enumerate(rows):
    d = data['08'] - data['Base']
    with np.errstate(invalid='ignore', divide='ignore'):
        dp = np.where(np.abs(data['Base']) > 0.01 * np.nanmax(np.abs(data['Base'])), d / data['Base'] * 100, np.nan)
    for j, (arr, unit) in enumerate(((d, f'Δ {lab}'), (dp, 'Δ (%)'))):
        ax = axes[i, j]; lim = sym_lim(arr, q=98)
        pc = draw_map(ax, arr, cmap='RdBu_r', vmin=-lim, vmax=lim)
        fig.colorbar(pc, ax=ax, shrink=0.85, label=unit)
        tag(ax, f'({LETTERS[2 * i + j]}) Δ forcing (08 − Base) — {rt}', y=0.98, size=8)
fig.suptitle('Δ forcing alone (own colour scales): shading + wind effect in WAQ', fontsize=10)
plt.tight_layout(); save(fig, 'Compare3_maps_surface_forcing_only.png'); plt.show()

print('Surface layer, period mean: inside footprint / outside / whole lake')
for (rt, data, lab, _) in rows:
    print(f'\n{rt} ({lab})')
    out = {}
    for name, m in (('inside', FOOT & WET_MN), ('outside', ~FOOT & WET_MN), ('lake', WET_MN)):
        v = {r: float(np.nanmean(data[r][m])) for r in RUNS}
        out[name] = {**{RUNS[r]['label']: v[r] for r in RUNS},
                     **{f'{lab_d} (%)': (v[a] - v[b]) / v[b] * 100 for d, (a, b, lab_d) in DIFFS.items()}}
    print(pd.DataFrame(out).round(3).to_string())

### 12.3 Depth profiles averaged over the FPV footprint and over the whole lake
For each day and layer: volume-weighted mean over the wet segments of the area,
`Σ c·V / Σ V`, with the daily volumes of that run. The depth axis uses nominal layer mid-depths
(baseline June–August median thickness of each layer in the area).

In [ ]:
def area_profile(field, vg, mask):
    m = mask[None, None] & np.isfinite(field) & (vg > 0)
    w = np.where(m, vg, 0.0)
    num = np.where(m, field * w, 0.0).sum(axis=(2, 3))
    den = w.sum(axis=(2, 3))
    with np.errstate(invalid='ignore', divide='ignore'):
        return np.where(den > 0, num / den, np.nan)              # (day, layer)

def area_mids(mask):
    th = THICK['Base'][summer][:, in_ks]
    h = []
    for j, k in enumerate(KS):
        segs = np.flatnonzero((POS[in_ks, 2] == k) & mask[POS[in_ks, 0], POS[in_ks, 1]])
        h.append(np.nanmedian(np.nanmean(th[:, segs], axis=0)) if segs.size else np.nan)
    h = np.nan_to_num(np.array(h))
    return np.cumsum(h) - h / 2

def draw_prof(ax, a, dep=None, **kw):
    pc = ax.pcolormesh(DAYS, dep, a.T, shading='nearest', **kw)
    if not ax.yaxis_inverted():
        ax.invert_yaxis()
    fmt_time(ax, 7)
    return pc

def area_figure(mask, name, fname):
    dep = area_mids(mask)
    rows = [(v, {r: area_profile(FIELD3[r][v], VGRID[r], mask) for r in RUNS}, lab, cmap)
            for v, lab, cmap in VARS12]
    keep = np.isfinite(rows[0][1]['Base']).any(axis=0)          # layers present in this area
    rows = [(rt, {r: a[:, keep] for r, a in d.items()}, lab, cmap) for rt, d, lab, cmap in rows]
    fig = plt.figure(figsize=(19, 6.5))
    six_columns(fig, rows, lambda ax, a, **kw: draw_prof(ax, a, dep=dep[keep], **kw), ylabel='Depth (m)')
    fig.suptitle(f'{name} — volume-weighted depth profiles', fontsize=11)
    save(fig, fname); plt.show()
    print(f'{name}: {CHL_PERIOD[0]:%d %b}–{CHL_PERIOD[1]:%d %b} mean, surface layer and whole-column '
          f'(volume-weighted)')
    sel = (DAYS >= CHL_PERIOD[0]) & (DAYS <= CHL_PERIOD[1])
    out = {}
    for rt, d, lab, _ in rows:
        for r in RUNS:
            f, vg = FIELD3[r][rt], VGRID[r]
            m = mask[None, None] & np.isfinite(f) & (vg > 0)
            col = np.where(m, f * vg, 0).sum(axis=(1, 2, 3)) / np.where(m, vg, 0).sum(axis=(1, 2, 3))
            out[(rt, RUNS[r]['label'])] = {'surface': float(np.nanmean(d[r][sel, 0])),
                                           'column': float(np.nanmean(col[sel]))}
    print(pd.DataFrame(out).T.round(3).to_string())

area_figure(FOOT & WET_MN, 'FPV footprint average', 'Compare3_profiles_footprint.png')
area_figure(WET_MN, 'Whole-lake average', 'Compare3_profiles_lake.png')

### 12.4 P3 (under the panels) and BIS (outside)
Four rows: P3 Chl a, BIS Chl a, P3 algae, BIS algae. The two stations of the same variable share their colour
scales, so the rows can be compared directly. Chl a here is the derived value (12.1) at both stations, so that P3
and BIS are treated identically; the check below compares it with BIS's own `.his` `Chlfa`.

In [ ]:
def station_field(st, run, v):
    s = ST[st]
    return FIELD3[run][v][:, [k - K_SURF for k in s['ks']], s['m'], s['n']]

rows = []
for v, lab, cmap in VARS12:
    for st in ('P3', 'BIS'):
        rows.append((f'{st} — {v}', {r: station_field(st, r, v) for r in RUNS}, lab, cmap))

# colour limits shared by the two stations of the same variable
fig = plt.figure(figsize=(19, 12))
gs_rows = []
for i in range(0, len(rows), 2):
    a, b = rows[i], rows[i + 1]
    both = {r: np.concatenate([a[1][r].ravel(), b[1][r].ravel()]) for r in RUNS}
    D = {d: np.concatenate([a[1][x].ravel() - a[1][y].ravel(), b[1][x].ravel() - b[1][y].ravel()])
         for d, (x, y, _) in DIFFS.items()}
    lims = (abs_lim(*both.values()), sym_lim(*D.values(), q=98))
    gs_rows += [(a, lims), (b, lims)]

gs = gridspec.GridSpec(len(gs_rows), 9, figure=fig, width_ratios=[1, 1, 1, 0.05, 0.22, 1, 1, 1, 0.05],
                       wspace=0.08, hspace=0.20, left=0.05, right=0.96, top=0.94, bottom=0.05)
p = 0
for i, ((rt, data, lab, cmap), ((vmin, vmax), dl)) in enumerate(gs_rows):
    dep = ST[rt.split(' — ')[0]]['mid']
    axes = []
    for c, (r, title) in enumerate(COLS12):
        ax = fig.add_subplot(gs[i, c], sharey=axes[0] if axes else None)
        pc = draw_prof(ax, data[r], dep=dep, cmap=cmap, vmin=vmin, vmax=vmax); axes.append(ax)
        tag(ax, f'({LETTERS[p]}) {title}', y=0.98, size=8); p += 1
    for c, (d, title) in enumerate(DCOLS12):
        x, y_, _ = DIFFS[d]
        ax = fig.add_subplot(gs[i, 5 + c], sharey=axes[0])
        pd_ = draw_prof(ax, data[x] - data[y_], dep=dep, cmap='RdBu_r', vmin=-dl, vmax=dl); axes.append(ax)
        tag(ax, f'({LETTERS[p]}) {title}', y=0.98, size=8); p += 1
    for ax in axes[1:]:
        plt.setp(ax.get_yticklabels(), visible=False)
    axes[0].set_ylabel(f'{rt}\nDepth (m)', fontsize=9)
    if i < len(gs_rows) - 1:
        for ax in axes:
            plt.setp(ax.get_xticklabels(), visible=False)
    fig.colorbar(pc, cax=fig.add_subplot(gs[i, 3]), label=lab).ax.tick_params(labelsize=7)
    fig.colorbar(pd_, cax=fig.add_subplot(gs[i, 8]), label=f'Δ {lab}').ax.tick_params(labelsize=7)
fig.suptitle(f'P3 (covered fraction {ST["P3"]["cover"]:.2f}) vs BIS (covered fraction {ST["BIS"]["cover"]:.2f}) '
             '— depth profiles, common colour scales per variable', fontsize=11)
save(fig, 'Compare3_profiles_P3_BIS.png'); plt.show()

sel = (DAYS >= CHL_PERIOD[0]) & (DAYS <= CHL_PERIOD[1])
out = {}
for rt, data, lab, _ in rows:
    v = {r: float(np.nanmean(data[r][sel, 0])) for r in RUNS}
    out[rt] = {**{RUNS[r]['label']: v[r] for r in RUNS},
               **{f'{l} (%)': (v[a] - v[b]) / v[b] * 100 for d, (a, b, l) in DIFFS.items()}}
print(f'Surface layer, {CHL_PERIOD[0]:%d %b}–{CHL_PERIOD[1]:%d %b} mean')
print(pd.DataFrame(out).T.round(3).to_string())

chk = {r: (np.nanmean(station_field('BIS', r, 'Chl a')[sel, 0]), top('BIS', r, 'Chlfa')[sel].mean()) for r in RUNS}
print('\nCheck at BIS surface: derived Chl a vs .his Chlfa (period mean, µg/L)')
for r, (dv, hv) in chk.items():
    print(f'   {RUNS[r]["label"]:34s} derived {dv:7.3f}   .his {hv:7.3f}')

## 13. Summary tables
- **13.1** Monthly table per station (layout of the earlier Section 8): Baseline, 08, FPV 02 and the three Δ rows.
- **13.2** Compact June–August attribution table across the three stations.
Both are exported as CSV to `RESULTS_DIR`.

In [ ]:
TABLE_VARS = [   # (variable, unit, layer, function(st, run) -> daily series)
    ('Top-cell thickness', 'm',     'Surface',   lambda st, r: top(st, r, 'Depth')),
    ('Water depth',        'm',     'Column',    lambda st, r: water_depth(st, r)),
    ('Temperature',        'degC',  'Surface',   lambda st, r: top(st, r, 'Temp')),
    ('DO',                 'g/m3',  'Surface',   lambda st, r: top(st, r, 'OXY')),
    ('DO',                 'g/m3',  'Bottom',    lambda st, r: bottom(st, r, 'OXY')),
    ('DO',                 'g/m3',  'Depth-avg', lambda st, r: depth_avg(st, r, 'OXY')),
    ('Algae',              'gC/m3', 'Surface',   lambda st, r: top(st, r, 'Algae')),
    ('Algae',              'gC/m2', 'Column',    lambda st, r: column_int(st, r, 'Algae')),
    ('Chl a',              'ug/L',  'Surface',   lambda st, r: top(st, r, 'Chlfa')),
    ('Secchi',             'm',     'Averaged',  lambda st, r: daily(his_raw(HIS[r], f'{ST[st]["his"]}_averaged',
                                                                     'SecchiDept')) if ST[st]['his'] else
                                                                     pd.Series(np.nan, index=DAYS)),
    ('NO3',                'gN/m3', 'Depth-avg', lambda st, r: depth_avg(st, r, 'NO3')),
    ('NH4',                'gN/m3', 'Depth-avg', lambda st, r: depth_avg(st, r, 'NH4')),
    ('PO4',                'gP/m3', 'Surface',   lambda st, r: top(st, r, 'PO4')),
    ('PO4',                'gP/m3', 'Depth-avg', lambda st, r: depth_avg(st, r, 'PO4')),
    ('Si',                 'gSi/m3', 'Depth-avg', lambda st, r: depth_avg(st, r, 'Si')),
    ('RadAve',             'W/m2',  'Surface',   lambda st, r: top(st, r, 'RadAve')),
    ('VWind',              'm/s',   'Surface',   lambda st, r: top(st, r, 'VWind')),
]
PERIODS = {m.strftime('%b %y'): (m, m + pd.offsets.MonthEnd(0)) for m in pd.date_range(SIM_START, SIM_END, freq='MS')}
PERIODS['Apr-Aug'] = (SIM_START, SIM_END)
PERIODS['Jun-Aug'] = SUMMER

def period_means(s):
    return {p: s.loc[a:b].mean() for p, (a, b) in PERIODS.items()}

all_rows = []
for st in ST:
    for var, unit, layer, fn in TABLE_VARS:
        S = {r: fn(st, r) for r in RUNS}
        if not any(S[r].notna().any() for r in RUNS):
            continue
        out = {RUNS[r]['label']: S[r] for r in RUNS}
        for d, (a, b, lab) in DIFFS.items():
            out[lab] = S[a] - S[b]
        for row_label, s in out.items():
            all_rows.append({'Station': st, 'Variable': var, 'Unit': unit, 'Layer': layer, 'Row': row_label,
                             **period_means(s)})
monthly = pd.DataFrame(all_rows).set_index(['Station', 'Variable', 'Unit', 'Layer', 'Row'])
monthly.to_csv(RESULTS_DIR / 'FPV_attribution_monthly_by_station.csv')
with pd.option_context('display.width', 250, 'display.max_rows', 2000):
    for st in ST:
        print('\n' + '=' * 120 + f'\n{st} — monthly means\n' + '=' * 120)
        print(monthly.loc[st].round(3).to_string())

In [ ]:
rows = []
for st in ST:
    for var, unit, layer, fn in TABLE_VARS:
        if var in ('RadAve', 'VWind'):
            continue
        S = {r: fn(st, r).loc[SUMMER[0]:SUMMER[1]].mean() for r in RUNS}
        if not np.isfinite(S['Base']):
            continue
        row = {'Station': st, 'Variable': f'{var} {layer} ({unit})', 'Baseline': S['Base']}
        for d, (a, b, lab) in DIFFS.items():
            row[f'{d} (abs)'] = S[a] - S[b]
            row[f'{d} (%)'] = (S[a] - S[b]) / abs(S[b]) * 100 if S[b] else np.nan
        rows.append(row)
compact = pd.DataFrame(rows).set_index(['Station', 'Variable'])
compact.to_csv(RESULTS_DIR / 'FPV_attribution_JunAug_compact.csv')
print('June–August means: Baseline and the three differences (abs and % of the reference run)')
with pd.option_context('display.width', 250, 'display.max_rows', 500):
    print(compact.round(3).to_string())
print(f'\nCSV files written to {RESULTS_DIR}')

In [ ]:
print('Checks summary (Section 4)')
for name, (ok, detail) in RESULTS.items():
    print(f"  [{'PASS' if ok else 'FAIL'}] {name}")
print(f'{sum(ok for ok, _ in RESULTS.values())} PASS, {sum(not ok for ok, _ in RESULTS.values())} FAIL')

## 14. Reading the results

**How to read the attribution**
- Δ total = Δ forcing + Δ hydro, exactly (same days, same layers).
- **Δ forcing** is the direct effect of the shade and the wind reduction on WAQ. In this model the only light
  pathway is BLOOM's light (energy) limit, and wind only acts through reaeration and similar processes
  (check the active processes in the `.lsp` if this matters).
- **Δ hydro** contains everything the FPV FLOW run changes: water level (less evaporation under the panels),
  temperature (less heat input), and vertical mixing. These three are not separated by these runs.

**Things to check before interpreting a number**
1. Section 4 must be all PASS (otherwise 08 is not "baseline FLOW + FPV forcing").
2. Top cell and `MinDepth` (Sections 5 and 11.4). In a z-layer grid, a water-level difference of a few
   centimetres can double or halve the top cell, and a cell thinner than `MinDepth` has its processes switched
   off. A large Δ hydro that follows the top-cell thickness rather than the footprint is a grid effect triggered
   by the (real) level difference, not a direct effect of the panels.
3. Spatial pattern (Section 11). A change confined to the footprint is local; the same change inside and outside
   the footprint is reservoir-wide (horizontal exchange, or a reservoir-wide driver such as the level).
4. Stations. CAB sits on the edge of the footprint and BIS outside it; P3 is the cleanest station for a local
   shading effect.
5. Chl a vs algal biomass. P3 and the maps use total algal carbon; `Chlfa` (CAB, BIS) also depends on BLOOM's
   chlorophyll-to-carbon ratios, so the two can differ in relative change.
6. Daily means include the 00:00 BLOOM step, where the thin top cell spikes; this affects absolute values
   more than differences between runs.

**Observed when these runs were first compared (12 October 2026; recomputed above, check they still hold)**
- 08 vs Baseline at the stations: surface Chl a −0 to −5 %, column Chl a −0 to −4 %; spatially −4.1 % surface
  algae inside the footprint vs −2.4 % outside, −2.6 % one layer down everywhere.
- FPV vs 08: surface algae +124 to +133 %, one layer down about +199 %, the same inside and outside the footprint;
  FPV water level 0.07–0.21 m higher (April → August); baseline top cell below `MinDepth` on 61 % of August days
  at CAB and BIS, FPV never.
- Light (`Limit e`) never binds in the top layer at CAB or BIS in any run; it binds in every layer below.